# 05.03 · Consuming MCP servers with MCPAdapter

**Module 05 · MCP & External Tools** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 05.02 (MCP fundamentals, top-level await), Lesson 02.03 (`create_agent`). |
| 🔑 Keys | `GROQ_API_KEY` (optional; the docs server needs no key) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Load** tools from an MCP server with `MCPAdapter` and `list_tools()`.
2. **Run** an agent over MCP tools with `create_agent` and `await agent.ainvoke(...)` (MCP tools are async-only).
3. **Read** MCP tool results: content blocks, `artifact`, `status="error"`, and the server metadata.
4. **Combine** several servers (prefixed tool names) and avoid name clashes.
5. **Explain** the connection lifecycle: sessions, caching, and when tools reconnect.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [LangChain: MCP overview](https://docs.langchain.com/oss/python/langchain/mcp/index) | `MCPAdapter`, targets, the quickstart |
| [LangChain: MCP connections](https://docs.langchain.com/oss/python/langchain/mcp/connections) | URLs, paths, configs, multiple servers, lifecycle |
| [LangChain: MCP tools](https://docs.langchain.com/oss/python/langchain/mcp/tools) | Results, artifacts, errors, metadata |
| [Migrate from langchain-mcp-adapters](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters) | Old tutorials use `MultiServerMCPClient`: the mapping to `MCPAdapter`, and what is *not yet wrapped* |
| [Docs by LangChain MCP server](https://docs.langchain.com/mcp) · [Use these docs with MCP](https://docs.langchain.com/use-these-docs) | The free, read-only docs server we use |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langchain[mcp]>=1.4.4,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "fastmcp",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/05-mcp-and-external-tools/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import logging
import warnings

# (a) Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# (b) `langchain.mcp` is beta and says so once per process (LangChainBetaWarning). Read it once in 05.02; muted after.
from langchain_core._api import LangChainBetaWarning  # noqa: E402  (import first: langchain_core adds its own filter)

warnings.filterwarnings("ignore", message=".*langchain.mcp.* is in beta", category=LangChainBetaWarning)
# (c) In-process FastMCP servers log every failing tool call with a long boxed traceback. The client still gets
#     the error as a ToolMessage(status="error"); we show it ourselves. Remove these lines to see the server's view.
import fastmcp  # noqa: E402,F401  (importing fastmcp configures its logger, so set the level after the import)

logging.getLogger("fastmcp").setLevel(logging.CRITICAL)

In [ ]:
# Step 5: Imports. docs_standin is an offline copy of the docs server (86 saved sections, same output format).
import tempfile
import time

from langchain.agents import create_agent
from langchain.mcp import MCPAdapter
from langchain.messages import AIMessage, HumanMessage, ToolMessage

from mastery.mcp import check_sources, extract_urls, serve_http, tool_info, urls_from_messages
from mastery.mcp_servers import docs_standin
from mastery.mcp_servers.notes_server import build_server
from mastery.testing import ScriptedChatModel

DOCS_MCP = "https://docs.langchain.com/mcp"
standin = docs_standin.build_server()
NOTES_DIR = tempfile.mkdtemp(prefix="m05-notes-")
print("stand-in pages:", len(docs_standin.load_snapshot()))

**You should now see** `stand-in pages: 86`.

---
## Part 1 · From server to LangChain tools

`MCPAdapter(target)` takes a URL (`"https://..."`), a path to a server script (stdio), a FastMCP server object (in-memory), or a config dict for several servers. Open it with `async with`, call `await adapter.list_tools()`, and you get ordinary LangChain `BaseTool`s.

In [ ]:
# 🧪 Offline: the stand-in (in-memory). Same tool name and output format as the real docs server.
async with MCPAdapter(standin) as adapter:
    docs_tools = await adapter.list_tools()

for row in tool_info(docs_tools):
    print(row)
search_docs = docs_tools[0]
print("\nargs:", search_docs.args)

**You should now see** one dict for `search_docs_by_lang_chain` with `read_only: True`, the server name `Docs by LangChain (offline stand-in)`, and `args: {'query': {...}}`. `tool_info` reads the MCP metadata LangChain keeps on each tool (`tool.metadata["mcp"]`).

In [ ]:
# 🧪 Offline: call the tool directly. MCP tools are async-only, so use ainvoke.
try:
    search_docs.invoke({"query": "checkpointer"})
except NotImplementedError as exc:
    print("invoke  →", exc)

call = {"name": search_docs.name, "args": {"query": "python short-term memory checkpointer"}, "id": "t1", "type": "tool_call"}
result = await search_docs.ainvoke(call)
print("ainvoke →", type(result).__name__, "| status:", result.status)
print(result.content[0]["text"][:300])
print("\nURLs:", urls_from_messages([result])[:4])

**You should now see** `invoke → StructuredTool does not support sync invocation.`, then `ainvoke → ToolMessage | status: success`, the first result block (`Title: ... Link: https://docs.langchain.com/oss/python/langchain/short-term-memory... Page: ... Content: ...`), and up to four docs URLs.

Two things to remember:
- **Async-only.** Agents with MCP tools need `await agent.ainvoke(...)` (or `astream`). Middleware with `wrap_*` hooks needs async versions too (`awrap_tool_call`); Module 04's `AuditLog` and `RoleToolFilter` now have both (decision D-013).
- **Content is a list of blocks.** Called with a tool-call dict, the tool returns a `ToolMessage` whose `content` is `[{"type": "text", "text": ...}]`; structured data (if the server sends it) is in `result.artifact["structured_content"]`.

> 🌐 Needs internet, no key, no tokens. Offline or behind a firewall? The cell prints a ⚠️ and the lesson continues with the offline stand-in.

In [ ]:
# 🌐 The real docs server: same calls, live data. Falls back to the stand-in if unreachable.
try:
    async with MCPAdapter(DOCS_MCP) as adapter:
        live_docs_tools = await adapter.list_tools()
    start = time.perf_counter()
    live = await live_docs_tools[0].ainvoke({**call, "name": live_docs_tools[0].name})
    print(f"{len(live_docs_tools)} tools; search took {time.perf_counter() - start:.2f}s")
    print(live.content[0]["text"][:300])
except Exception as exc:
    live_docs_tools = docs_tools
    print(f"⚠️ {DOCS_MCP} unreachable ({type(exc).__name__}). Using the offline stand-in from here on.")

**You should now see** `3 tools; search took 0.xs` and a result block in the same shape as the stand-in's, or the ⚠️ fallback line.

---
## Part 2 · An agent over MCP tools

In [ ]:
# 🧪 Offline: a scripted model searches the stand-in and cites a Link from the result.
def scripted_docs_agent():
    """A fresh script each time: a scripted model's replies are used up as they're read."""
    script = ScriptedChatModel(responses=[
        AIMessage("", tool_calls=[{"name": "search_docs_by_lang_chain", "args": {"query": "python add short-term memory checkpointer"}, "id": "s1"}]),
        "Pass a checkpointer to create_agent (e.g. InMemorySaver) and reuse a thread_id. "
        "Source: https://docs.langchain.com/oss/python/langchain/short-term-memory#usage",
    ])
    return create_agent(script, tools=docs_tools, system_prompt="Answer from the docs tools. Cite Link URLs from results.")


try:
    scripted_docs_agent().invoke({"messages": [HumanMessage("How do I add short-term memory?")]})
except NotImplementedError as exc:
    print("invoke  →", type(exc).__name__)

out = await scripted_docs_agent().ainvoke({"messages": [HumanMessage("How do I add short-term memory?")]})
answer = out["messages"][-1].text
grounded, invented = check_sources(extract_urls(answer), urls_from_messages(out["messages"]))
print(answer)
print("grounded:", grounded, "| invented:", invented)

**You should now see** `invoke → NotImplementedError`, then the answer with its source URL, `grounded: ['https://docs.langchain.com/oss/python/langchain/short-term-memory#usage']` and `invented: []`. The cited URL appears in a tool result, so it is grounded.

> 💸 Uses a few hundred to a few thousand tokens of your free-tier quota (docs results are long: about 2–5k input tokens per search). Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: a real model over the docs tools (live server, or the stand-in if you're offline).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_agent = create_agent(get_model("fast", temperature=0), tools=live_docs_tools[:1],
                              system_prompt="Answer in 3 sentences from search_docs_by_lang_chain results only. "
                                            "Add 'python' to queries. End with 'Sources:' and Link URLs from the results.")
    out = await live_agent.ainvoke({"messages": [HumanMessage("How do I stream tokens from an agent?")]})
    print(out["messages"][-1].text)
    final_urls = extract_urls(out["messages"][-1].text)
    print("\ngrounded/invented:", check_sources(final_urls, urls_from_messages(out["messages"])))
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) a short answer about `stream_mode="messages"` and `Sources:` with docs URLs, then the grounded/invented split, usually with nothing invented.

### What just happened?

The adapter turned each MCP tool into a LangChain tool whose `_arun` opens an MCP session and sends `tools/call`. `create_agent` doesn't know or care that the tool is remote. The only visible differences: async-only, and content arrives as blocks.

---
## Part 3 · Several servers at once

Two ways:
1. **One adapter per server**, then combine the tool lists. Simple; you choose names.
2. **One config** in the standard `mcpServers` format (the same JSON IDEs use). Tool names get the server key as a prefix (`docs_search_docs_by_lang_chain`), so two servers can both have a `search` tool.

In [ ]:
# 🧪 Offline: way 1, two in-memory servers combined.
async with MCPAdapter(build_server(NOTES_DIR)) as adapter:
    notes_tools = await adapter.list_tools()
combined = docs_tools + notes_tools
names = [t.name for t in combined]
print(names)
assert len(names) == len(set(names)), "two tools share a name: use a config (prefixes) instead"

**You should now see** `search_docs_by_lang_chain` plus the five notes tools, and no assertion error.

> 🌐 Needs internet, no key, no tokens. Offline or behind a firewall? The cell prints a ⚠️ and the lesson continues with the offline stand-in.

In [ ]:
# 🌐 Way 2: the docs server (remote HTTP) + the notes server (local HTTP) in one standard config.
with serve_http(env={"MASTERY_NOTES_DIR": NOTES_DIR}) as notes_url:
    config = {"mcpServers": {"docs": {"url": DOCS_MCP}, "notes": {"url": notes_url}}}
    try:
        async with MCPAdapter(config) as adapter:
            grouped = await adapter.list_tools()
        for t in grouped:
            print(t.name)
    except Exception as exc:
        print(f"⚠️ Could not load both servers ({type(exc).__name__}). Offline? Way 1 above works without a network.")

**You should now see** eight names, each prefixed with its config key: `docs_search_docs_by_lang_chain`, `docs_query_docs_filesystem_docs_by_lang_chain`, `docs_submit_feedback`, `notes_save_note`, ... (With a config, `tool.metadata["mcp"]["server"]` names the router FastMCP builds, `MCPRouter`, not each server: the prefix is how you tell them apart.) (If one server is down, the whole config fails to load: in production, load servers separately so one outage doesn't take all tools away.)

---
## Part 4 · The connection lifecycle

- `async with MCPAdapter(...)` opens a session for **discovery**. `list_tools()` caches the result (`cache_mode="use"`); pass `"refresh"` to re-fetch, `"bypass"` to skip the cache.
- The tools keep a reference to the client, so they **still work after the `async with` block**. Each call opens its own short session, so tools can be called concurrently.
- Cost of that design: one connection setup per call. Local and in-memory servers don't notice; for a slow remote server, keep calls few and focused.

In [ ]:
# 🧪 Offline: caching, and tools that outlive the context.
adapter = MCPAdapter(build_server(NOTES_DIR))
async with adapter:
    start = time.perf_counter(); first = await adapter.list_tools(); t_first = time.perf_counter() - start
    start = time.perf_counter(); again = await adapter.list_tools(); t_again = time.perf_counter() - start
    fresh = await adapter.list_tools(cache_mode="refresh")
print(f"first {t_first * 1000:.1f} ms, cached {t_again * 1000:.2f} ms, refreshed: {len(fresh)} tools")

save_note = next(t for t in first if t.name == "save_note")
after = await save_note.ainvoke({"name": "save_note", "args": {"title": "Lifecycle", "body": "Tools outlive async with."}, "id": "l1", "type": "tool_call"})
print("call after exit →", after.status, after.artifact["structured_content"])

**You should now see** two short timings (the cached listing skips the `tools/list` request, so it is usually faster; in-memory both are tiny), `refreshed: 5 tools`, and `call after exit → success {'id': 'n-...', 'title': 'Lifecycle', 'tags': []}`.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `NotImplementedError: StructuredTool does not support sync invocation.` | `agent.invoke` / `tool.invoke` with MCP tools | `await agent.ainvoke(...)` / `await tool.ainvoke(...)` |
| `NotImplementedError: Asynchronous implementation of awrap_tool_call is not available` | Your middleware only defines sync `wrap_tool_call` | Add `async def awrap_tool_call` (see `mastery/middleware/audit.py`) |
| `ValueError` about the target, for `"server.py"` | A string target must be an `http(s)` URL | `Path("server.py")` for stdio |
| Results about JavaScript | The docs server covers JS and Python | Put "python" in the query, or filter on `/oss/python/` links |
| `ToolMessage(status="error")` with `Error calling tool ...` | The server reported an error (`isError`) | Normal: the model sees it and can retry; inspect the text |
| `ConnectError`, timeouts | Server down, offline, proxy | Use the stand-in; on corporate Windows networks see 00.02 Step 4b |
| Duplicate tool names | Two servers with the same tool | Use an `mcpServers` config (prefixes), or rename |
| An invented URL counts as "grounded" with `response_format=ToolStrategy(...)` | ToolStrategy adds a `ToolMessage` (`Returning structured response: ...`) that repeats the model's own sources | `urls_from_messages(messages, skip_tools={"<YourSchema>"})` (used in Project 05) |

## ⚠️ Production notes

> ⚠️ **Production note: remote tools fail like remote services.**
> Set timeouts, handle `status="error"` results, and load servers independently so one outage doesn't remove every tool. Log the server name and version (`tool.metadata["mcp"]["server"]`) with each call; it's your first clue when a server changes underneath you.

> ⚠️ **Production note: pin what the model sees.**
> A server can change its tool list or descriptions at any time. Record a fingerprint of the tools you reviewed (`mastery.mcp.tool_fingerprint`, Lesson 05.05) and alert when it changes.

## 🏋️ Exercises

1. **(Easy)** Search the stand-in for `"python interrupt resume Command"`. Which pages come back?
2. **(Medium)** Use `query_docs_filesystem_docs_by_lang_chain` on the live server with the command `ls /oss/python/langchain/mcp`. What pages exist?
3. **(Medium)** Add the notes tools to the scripted docs agent and script a `search_notes` call. Run it with `ainvoke`.
4. **(Stretch)** Write a `docs_tools_or_standin()` coroutine that returns live tools, or the stand-in's within 5 seconds if the server is unreachable.

<details>
<summary><b>Show solutions</b></summary>

**1.** `await search_docs.ainvoke({"name": search_docs.name, "args": {"query": "python interrupt resume Command"}, "id": "e1", "type": "tool_call"})` → mostly `langgraph/interrupts` sections and the human-in-the-loop page.

**2.** Pick the tool by name from `live_docs_tools` and call it with `{"command": "ls /oss/python/langchain/mcp"}`: index, connections, tools, auth, and more pages.

**3.** `create_agent(script, tools=docs_tools + notes_tools)` with a scripted `AIMessage(tool_calls=[{"name": "search_notes", "args": {"query": "lifecycle"}, "id": "n1"}])`.

**4.** `async with asyncio.timeout(5): async with MCPAdapter(DOCS_MCP) as a: return await a.list_tools()`, inside `try/except Exception`, falling back to `MCPAdapter(docs_standin.build_server())`.
</details>

## ✅ Checkpoint

- [ ] I can load tools from a URL, a server object and a config with `MCPAdapter`.
- [ ] I run MCP agents with `await agent.ainvoke(...)`.
- [ ] I can read content blocks, `artifact` and `status="error"`.
- [ ] I can combine servers without name clashes.
- [ ] I can explain caching and per-call sessions.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

On the [MCP connections page](https://docs.langchain.com/oss/python/langchain/mcp/connections), check which targets `MCPAdapter` accepts today and whether SSE or WebSocket support changed. On the [migration page](https://docs.langchain.com/oss/python/migrate/langchain-mcp-adapters), check the "Prompts and resources" section: still *no wrapper yet* (2026-10-12)? Log it in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `MCPAdapter(target)` + `await list_tools()` → LangChain tools; works with `create_agent` unchanged.
- MCP tools are async-only: `ainvoke`, and async middleware hooks.
- Results are content blocks (+ `artifact`); server errors are `status="error"` messages.
- Configs combine servers with prefixed names; tools outlive the adapter context.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 05.04 · Building your own MCP server with FastMCP** → [`05-04-build-an-mcp-server.ipynb`](05-04-build-an-mcp-server.ipynb).

[Module 05 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)